In [8]:
# Debug version - run this first
import pandas as pd
import requests
from datetime import datetime, timedelta
import os
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("ALPHA_VANTAGE_API_KEY")

# Check API key
print(f"API Key loaded: {API_KEY is not None}")
print(f"API Key length: {len(API_KEY) if API_KEY else 'None'}")

# Test with one ticker
ticker = "AAPL"
print(f"\n🔍 Testing {ticker}")

r = requests.get("https://www.alphavantage.co/query", params={
    "function": "TIME_SERIES_DAILY_ADJUSTED",
    "symbol": ticker,
    "outputsize": "compact",
    "datatype": "json",
    "apikey": API_KEY
})

print(f"Status code: {r.status_code}")
print(f"Response keys: {list(r.json().keys())}")
print(f"Full response: {r.json()}")

API Key loaded: True
API Key length: 16

🔍 Testing AAPL
Status code: 200
Response keys: ['Information']
Full response: {'Information': 'Thank you for using Alpha Vantage! This is a premium endpoint. You may subscribe to any of the premium plans at https://www.alphavantage.co/premium/ to instantly unlock all premium endpoints'}


In [9]:
import pandas as pd
import requests
from datetime import datetime, timedelta
from time import sleep
from dotenv import load_dotenv
import os

load_dotenv()
API_KEY = os.getenv("ALPHA_VANTAGE_API_KEY")
TICKERS = ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'AMZN']
OLD_PATH = "../data/raw/multi_stock_merged.csv"

df_old = pd.read_csv(OLD_PATH, parse_dates=['Date'])
since_date = (datetime.now() - timedelta(days=30)).date()

all_new = []
for ticker in TICKERS:
    print(f"📥 {ticker}")
    r = requests.get("https://www.alphavantage.co/query", params={
        "function": "TIME_SERIES_DAILY",  # Free tier version
        "symbol": ticker,
        "outputsize": "compact",
        "datatype": "json",
        "apikey": API_KEY
    })
    
    data = r.json().get("Time Series (Daily)", {})
    if not data:
        print(f"❌ No data for {ticker}")
        continue
        
    df = pd.DataFrame.from_dict(data, orient="index")
    df = df.rename(columns={
        "1. open": "Open", "2. high": "High", "3. low": "Low",
        "4. close": "Close", "5. volume": "Volume"  # Note: column 5, not 6
    })[["Open", "High", "Low", "Close", "Volume"]].astype(float)
    
    df.index = pd.to_datetime(df.index)
    df = df[df.index.date >= since_date]
    df["Ticker"] = ticker
    df = df.reset_index().rename(columns={"index": "Date"})
    all_new.append(df)
    sleep(12)

# Combine and save
df_new = pd.concat(all_new, ignore_index=True)
df_combined = pd.concat([df_old, df_new], ignore_index=True)
df_combined = df_combined.drop_duplicates(subset=["Date", "Ticker"], keep="last")
df_combined = df_combined.sort_values(by=["Ticker", "Date"])
df_combined.to_csv("../data/raw/multi_stock_merged_updated.csv", index=False)
print("✅ Done")

📥 AAPL
📥 MSFT
📥 NVDA
📥 GOOGL
📥 AMZN
✅ Done


remove spy data

In [10]:
# Remove SPY
import pandas as pd

# Load your CSV
df = pd.read_csv("../data/raw/multi_stock_merged_updated.csv")

# Remove SPY rows
df_cleaned = df[df['Ticker'] != 'SPY']

print(f"Removed {len(df) - len(df_cleaned)} SPY rows")
print(f"Remaining shape: {df_cleaned.shape}")

# Save cleaned data
df_cleaned.to_csv("../data/raw/multi_stock_merged_updated.csv", index=False)

Removed 2908 SPY rows
Remaining shape: (14570, 7)
